In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

In [8]:
# 1. 定义参数
# OIH (油服), RKH (现已多为无数据或更名), RTH (零售)
# 建议使用目前流动性更好的相同主题代换：OIH, KBE(银行), RTH
tickers = ['OIH', 'KBE', 'RTH'] 
start_date = '2005-01-01'
end_date = '2024-12-31'
rf_annual = 0.04

print(f"正在下载数据: {tickers}...")

# 2. 下载数据
raw_data = yf.download(tickers, start=start_date, end=end_date)['Close']

/tmp/ipykernel_269257/3203155836.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, start=start_date, end=end_date)['Close']
[*********************100%***********************]  3 of 3 completed

正在下载数据: ['OIH', 'KBE', 'RTH']...


In [9]:
# 3. 自动清洗：剔除下载失败或全为空值的列
data = raw_data.dropna(axis=1, how='all')
actual_tickers = data.columns.tolist()

if len(actual_tickers) == 0:
    print("错误：未能下载到任何有效数据，请检查网络或代码符号。")
  
print(f"有效参与计算的资产: {actual_tickers}")

有效参与计算的资产: ['KBE', 'OIH', 'RTH']


In [10]:
# 4. 计算收益率并剔除含有空值的行（确保矩阵对齐）
returns = data.pct_change().dropna()
returns

Ticker,KBE,OIH,RTH
Date,,,
2005-11-16,-0.004108,0.032613,0.006569
2005-11-17,0.011589,0.007120,0.010688
2005-11-18,0.004078,-0.001497,0.002135
2005-11-21,0.008315,0.032572,-0.000405
2005-11-22,0.004986,0.022187,0.006903
...,...,...,...
2024-12-23,0.001946,0.005971,-0.001225
2024-12-24,0.008452,0.010270,0.011470
2024-12-26,0.005528,0.005272,0.000785


In [11]:
# 5. 计算超额收益 (Excess Returns)
rf_daily = rf_annual / 252
excess_ret = returns - rf_daily

In [12]:
# 6. 计算年化均值向量 M 和协方差矩阵 C
# M 形状为 (n, 1)，C 形状为 (n, n)
M = excess_ret.mean().values.reshape(-1, 1) * 252
C = excess_ret.cov().values * 252

print(M)
print(C)

[[0.0496561 ]
 [0.005661  ]
 [0.09243084]]
[[0.12304135 0.07918009 0.04235921]
 [0.07918009 0.16196207 0.03518603]
 [0.04235921 0.03518603 0.03671019]]


In [13]:
# 7. 计算凯利最优杠杆 F* = inv(C) * M
try:
    F = np.linalg.inv(C) @ M
except np.linalg.LinAlgError:
    print("错误：协方差矩阵奇异（不可逆），请尝试减少资产数量或增加时间范围。")

In [14]:
# 8. 计算最大年化复合增长率 g 和夏普比率 S
# 公式: g = r + (F.T @ M) / 2  或者 g = r + (F.T @ C @ F) / 2
variance_p = (F.T @ C @ F)[0][0]
g = rf_annual + variance_p / 2
S = np.sqrt(variance_p)

In [15]:
# --- 输出结果 ---
print("\n" + "="*40)
print("      Kelly Formula 投资组合分析")
print("="*40)

results_df = pd.DataFrame({
    'Annualized Excess Return (M)': M.flatten(),
    'Optimal Leverage (F*)': F.flatten()
}, index=actual_tickers)

print(results_df.round(4))

print("\n年化协方差矩阵 (C):")
print(pd.DataFrame(C, index=actual_tickers, columns=actual_tickers).round(4))

print("-" * 40)
print(f"投资组合预期最大年化增长率 (g): {g:.2%}")
print(f"投资组合预期夏普比率 (S):      {S:.4f}")
print("="*40)


      Kelly Formula 投资组合分析
     Annualized Excess Return (M)  Optimal Leverage (F*)
KBE                        0.0497                -0.5123
OIH                        0.0057                -0.4926
RTH                        0.0924                 3.5811

年化协方差矩阵 (C):
        KBE     OIH     RTH
KBE  0.1230  0.0792  0.0424
OIH  0.0792  0.1620  0.0352
RTH  0.0424  0.0352  0.0367
----------------------------------------
投资组合预期最大年化增长率 (g): 19.14%
投资组合预期夏普比率 (S):      0.5503
